# Model: Suction-Line Restriction (Binary Classification)

## Feature choice, informed by notebook 06's EDA

Per notebook 06: every signal was cleanly monotonic and extremely strongly separated
at every severity - RTU_REFG_SUCT_PRES, RTU_REFG_SUCT_TEMP, and capacity all showed
the largest effect sizes found anywhere in the Simulated-dataset EDA (Cohen's
d=4.439 even at the mildest severity gap, 1 vs 3 bar). This is the sixth and final
Simulated fault to model.

## Real, final test of the open generalization-pattern question

5 of 6 faults modeled so far: 3 stable (overcharge, condenser fouling, liquid-line
restriction), 1 gradual decline (evaporator fouling), 1 collapse (undercharge).
Neither signal strength nor severity shape explained the split. This fault has the
single strongest signal in the whole dataset - if strength alone mattered, this
should be maximally stable. If it degrades anyway, that would further rule out
signal strength as the explanation and leave the question fully open pending a
dedicated investigation.

In [1]:
import sys
from pathlib import Path

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import classification_report  # noqa: E402
from sklearn.model_selection import TimeSeriesSplit, train_test_split  # noqa: E402
from src.features.build_features import build_feature_table  # noqa: E402

table = build_feature_table(
    baseline_path="../data/raw/RTU_sim_baseline.csv",
    fault_paths={
        "suctionpipe01bar": "../data/raw/RTU_sim_suctionpipe01bar.csv",
        "suctionpipe03bar": "../data/raw/RTU_sim_suctionpipe03bar.csv",
        "suctionpipe06bar": "../data/raw/RTU_sim_suctionpipe06bar.csv",
        "suctionpipe09bar": "../data/raw/RTU_sim_suctionpipe09bar.csv",
    },
    pressure_temp_cols=("RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP"),
)

print(f"Feature table shape: {table.shape}")
print(f"\nLabel distribution:\n{table['label'].value_counts()}")
table.head()

Feature table shape: (368630, 6)

Label distribution:
label
1    305440
0     63190
Name: count, dtype: int64


,Datetime,label,source_file,RTU_REFG_SUCT_PRES_residual,RTU_REFG_SUCT_TEMP_residual,RTU_TOT_CAPA_ewma30_segmented_residual
0,2018-07-20 01:00:00,0,baseline,-9.972298e+03,0.704927,662.149504
1,2018-07-20 01:00:00,1,suctionpipe06bar,-4.600033e+06,-21.319037,-4920.368496
2,2018-07-20 01:00:00,1,suctionpipe01bar,-4.546423e+05,-1.170935,107.440504
3,2018-07-20 01:00:00,1,suctionpipe09bar,-7.786290e+06,-41.906317,-8802.063496
4,2018-07-20 01:00:00,1,suctionpipe03bar,-2.945735e+06,-12.616878,-2844.201496


## Evaluating suction-line restriction: both random-split and TimeSeriesSplit

In [2]:
feature_cols = [
    "RTU_REFG_SUCT_PRES_residual",
    "RTU_REFG_SUCT_TEMP_residual",
    "RTU_TOT_CAPA_ewma30_segmented_residual",
]

X_all = table[feature_cols].values
y_all = table["label"].values

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
rf_random = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_random.fit(X_tr_r, y_tr_r)
y_pred_r = rf_random.predict(X_te_r)

print("=== Random split ===")
print(classification_report(y_te_r, y_pred_r, target_names=["baseline", "suctionline"]))

tscv = TimeSeriesSplit(n_splits=5)
print("=== TimeSeriesSplit (5 folds) ===")
for fold_num, (train_idx, test_idx) in enumerate(tscv.split(X_all), start=1):
    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]

    fold_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    fold_model.fit(X_tr, y_tr)
    y_pred_fold = fold_model.predict(X_te)

    report = classification_report(y_te, y_pred_fold, target_names=["baseline", "suctionline"], output_dict=True)
    print(f"Fold {fold_num}: baseline recall={report['baseline']['recall']:.2f}, "
          f"baseline precision={report['baseline']['precision']:.2f}, "
          f"suctionline recall={report['suctionline']['recall']:.2f}")

=== Random split ===
              precision    recall  f1-score   support

    baseline       0.91      0.82      0.86     12638
 suctionline       0.96      0.98      0.97     61088

    accuracy                           0.95     73726
   macro avg       0.94      0.90      0.92     73726
weighted avg       0.95      0.95      0.95     73726

=== TimeSeriesSplit (5 folds) ===
Fold 1: baseline recall=0.87, baseline precision=1.00, suctionline recall=1.00
Fold 2: baseline recall=0.77, baseline precision=1.00, suctionline recall=1.00
Fold 3: baseline recall=0.68, baseline precision=1.00, suctionline recall=1.00
Fold 4: baseline recall=0.60, baseline precision=1.00, suctionline recall=1.00
Fold 5: baseline recall=0.43, baseline precision=0.99, suctionline recall=1.00


## Suction-line restriction: gradual decline, matching evaporator fouling's pattern —
## NOT the stable pattern its extreme signal strength would predict

| | Random split | TS Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
|---|---|---|---|---|---|---|
| Baseline recall | 0.82 | 0.87 | 0.77 | 0.68 | 0.60 | 0.43 |
| Baseline precision | 0.91 | 1.00 | 1.00 | 1.00 | 1.00 | 0.99 |

Real, clear degradation trend (0.87 → 0.43, nearly halving) - matching evaporator
fouling's gradual-decline pattern, NOT condenser fouling/liquid-line restriction's
stability. This is a decisive result for the open question: **this fault has the
single strongest, cleanest effect sizes in the entire Simulated-dataset EDA
(Cohen's d=4.439), yet it degrades forward-in-time just as much as evaporator
fouling did.** This conclusively rules out signal strength as an explanation for
which faults are stable vs. degrading - the two strongest-signal faults in the
dataset (evaporator fouling, suction-line restriction) BOTH degrade, while
overcharge and liquid-line restriction (comparatively more moderate/threshold-shaped
signals) are stable.

**Precision stays essentially perfect throughout** (1.00, 1.00, 1.00, 1.00, 0.99) -
same as evaporator fouling's pattern (high, stable precision despite declining
recall). The model isn't becoming wrong, it's becoming more conservative/less
sensitive to baseline specifically, in later time periods, for both of these faults.

**A real pattern is now visible with all 6 faults modeled**: the two faults that
degrade (evaporator fouling, suction-line restriction) are BOTH the two "restriction
at the evaporator/suction side" and "evaporator fouling" - both directly involve the
evaporator/suction side of the refrigerant loop and produce very large capacity
swings. The three stable faults (overcharge, condenser fouling, liquid-line
restriction) do NOT directly degrade evaporator-side capacity as their primary
mechanism (overcharge's capacity effect was weak/non-monotonic; condenser fouling
degrades capacity only mildly; liquid-line restriction's capacity effect, while
real, was substantially reduced after stage-2 filtering). Undercharge (also
evaporator/suction-side, also degrades) fits this pattern too.

**Real, falsifiable hypothesis emerging**: faults that produce LARGE, direct swings
in capacity specifically may correlate with worse forward-in-time generalization -
possibly because capacity's own weather-driven variance (already established as
substantial, R²=0.767+ against RTU_OA_TEMP) interacts differently with a large fault
effect than a small one, even after residualization. This is a real, testable
pattern across all 6 data points now available, though still correlational, not
proven causal - worth a dedicated write-up in the log rather than further ad-hoc
testing today.

## Summary: suction-line restriction binary classifier — final of 6 Simulated faults

Real, clear forward-in-time degradation (baseline recall 0.87 -> 0.43 across folds),
matching evaporator fouling's pattern - despite having the single largest, cleanest
effect sizes in the entire Simulated-dataset EDA (Cohen's d=4.439). This conclusively
rules out signal strength as the explanation for stable-vs-degrading behavior.

**Real pattern found across all 6 faults**: the two degrading faults (undercharge,
evaporator fouling) plus this one (suction-line restriction) all directly involve
large capacity swings on the evaporator/suction side of the refrigerant loop. The
three stable faults (overcharge, condenser fouling, liquid-line restriction) do not
have capacity as their primary, large-magnitude effect. Falsifiable hypothesis:
large capacity swings specifically (not just any strong signal) correlate with worse
forward-in-time generalization, plausibly because capacity's substantial weather-
driven variance interacts poorly with a large fault effect even after
residualization. Correlational across 6 data points, not proven causal - a real
finding worth a dedicated investigation before the modeling phase is considered
complete.